In [1]:
from typing import List
import stim
from stimcirq import stim_circuit_to_cirq_circuit
import cirq
import openfermion as of
from encoded.code_extension import encoding_unitary_for_new_stabilizer
from encoded.utils import cirq_pauli_string_to_stim

In [2]:
n = 3
def extended_repetition_generators(n: int) -> List[stim.PauliString]:
    generators = []
    for i in range(n-1):
        pauli_str = 'I' * i + 'Z' * 2 + 'I' * (n - i - 2)
        assert len(pauli_str) == n
        generators.append(stim.PauliString(pauli_str))
    generators.append(stim.PauliString('X' * n))
    return generators

In [3]:
generators = extended_repetition_generators(n)
for generator in generators:
    print(generator)

+ZZ_
+_ZZ
+XXX


In [4]:
for gi in generators:
    for gj in generators:
        assert gi.commutes(gj)

In [5]:
def all_single_qubit_errs(n: int) -> List[stim.PauliString]:
    """Returns the set of all single-qubit Pauli errors."""

    errs = []
    for i in range(n):
        for p in ['X', 'Y', 'Z']:
            pauli_str = "I" * i + p + 'I' * (n - i - 1)
            assert len(pauli_str) == n
            errs.append(stim.PauliString(pauli_str))
    return errs

In [6]:
errors = all_single_qubit_errs(n)

In [7]:
number_false = 0
number_checked = 0
for i, ei in enumerate(errors):
    for j in range(i):
        number_checked += 1
        ej = errors[j]
        e = ei * ej
        commutators = []
        for generator in generators:
            comm = e.commutes(generator)
            commutators.append(comm)
        has_anticommuting_operator = any([not b for b in commutators])
        if has_anticommuting_operator:
            number_false += 1
        if not has_anticommuting_operator:
            print(f"{ei} * {ej} = {e}, {commutators} {has_anticommuting_operator} ")
print(f"{number_false}/{number_checked} operators anticommute.")

+_Z_ * +Z__ = +ZZ_, [True, True, True] False 
+__Z * +Z__ = +Z_Z, [True, True, True] False 
+__Z * +_Z_ = +_ZZ, [True, True, True] False 
33/36 operators anticommute.
